# README

- **Author**: RUEI-HSIANG, CHANG（張睿庠，B12303030；AI協助）
- **Created At**: 2026-11-10
- **Last Modified At**: 2026-11-10

## What does this file do?

以候選人唯一鍵candidate_key，將2026-11-08兩市210人研究暫納入版的G1–G6與既有G5／G6主群碼一對一連結。保留上游51欄與全部210人，只新增cluster_profile_id、cluster_profile_label兩欄。公職池由office_type分割為首長11／議員199；不是另對兩市各自分群。未修改G5／G6、其他背景值或資格規則。年齡缺值保留，不因年齡缺值刪人；學歷政策fallback旗標保留。

## What does this file take?

- Source Data Sets：
  - `data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv`；SHA-256 `35e3b85e407d4eb91e5c96043d06a407d21e070a5920032b26f74e916f883cbe`；唯一鍵candidate_key，210人。
  - `data/processed/tk_g5_g6_yu_provisional_all_assignments_20261108.csv`；SHA-256 `9b0fa7bd148d847dc3a8a778a435bc4ea90f0204a57ccd9dd46dfd3e44d7212f`；唯一鍵candidate_key，210人。
- 上游[資料與結果文件](../processed/taipei_kaohsiung_yu_provisional_20261108_documentation.ipynb)。兩檔共同欄位先逐格核對；禁止同名配對、未匹配或多對多合併。

## What does this file output?

- `data/processed/tk_g5_g6_characteristics_20261110.csv`：210列、53欄；一列一候選人，鍵candidate_key；role為既有分群的背景特徵分析輸入。
- [輸出文件](../processed/tk_g5_g6_characteristics_20261110_documentation.ipynb)；後續[finding](../../finding/taipei_kaohsiung_cluster_characteristics_20261110.ipynb)讀取同一精確檔案。
- Verification status / SHA-256：執行末格列出實際回讀檢查與輸出指紋；處理後文件與PROJECT.md記錄核對後結果。重跑只允許已有輸出與重算結果逐格相同，否則停止。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display, Markdown

def project_root():
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir(): return p
    raise FileNotFoundError('請從專案根目錄或其子目錄執行。')
ROOT=project_root()
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def checked_file(rel, expected):
    p=(ROOT/rel).resolve()
    assert p.is_relative_to((ROOT/'data/processed').resolve())
    assert p.is_file(), rel
    assert sha(p)==expected, f'輸入版本不同：{rel}'
    return p


In [2]:
PINS={'data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv': '35e3b85e407d4eb91e5c96043d06a407d21e070a5920032b26f74e916f883cbe', 'data/processed/tk_g5_g6_yu_provisional_all_assignments_20261108.csv': '9b0fa7bd148d847dc3a8a778a435bc4ea90f0204a57ccd9dd46dfd3e44d7212f'}
paths={p:checked_file(p,h) for p,h in PINS.items()}
source=pd.read_csv(paths['data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv'],dtype=str,keep_default_na=False)
assignment=pd.read_csv(paths['data/processed/tk_g5_g6_yu_provisional_all_assignments_20261108.csv'],dtype=str,keep_default_na=False)
KEY='candidate_key'
for frame in [source,assignment]:
    assert len(frame)==210 and frame[KEY].is_unique and frame[KEY].ne('').all()
assert source.shape[1]==51 and assignment.shape[1]==25
assert set(source[KEY])==set(assignment[KEY])
shared=[c for c in assignment.columns if c in source.columns]
pd.testing.assert_frame_equal(source.set_index(KEY)[[c for c in shared if c!=KEY]].sort_index(),
                              assignment.set_index(KEY)[[c for c in shared if c!=KEY]].sort_index())
print('兩份上游SHA、候選人鍵、全部共同欄位一致。')


兩份上游SHA、候選人鍵、全部共同欄位一致。


In [3]:
added=['cluster_profile_id','cluster_profile_label']
result=source.merge(assignment[[KEY,*added]],on=KEY,how='left',validate='one_to_one',sort=False)
assert len(result)==210 and result.shape[1]==53
assert result[added].notna().all().all()
pd.testing.assert_frame_equal(result[source.columns],source)
assert result.cluster_profile_id.eq('P'+result.G5_first_time_any_elected_office+result.G6_political_family).all()
assert result.cluster_profile_id.value_counts().to_dict()=={'P00':109,'P01':51,'P10':35,'P11':15}
assert result.office_type.value_counts().to_dict()=={'直轄市議員':199,'直轄市長':11}
assert result.county_city.value_counts().to_dict()=={'高雄市':106,'臺北市':104}
yu=result[result.eligibility_provisional_flag_20261108=='1']
assert len(yu)==1 and yu.iloc[0].candidate_name=='游智彬' and yu.iloc[0].cluster_profile_id=='P00'
assert result.G1_age_2026_minus_birth_year.eq('').sum()==21
assert result.G3_unpublished_lowest_flag.eq('1').sum()==15
print('210人一對一連結；已知值、缺值、資格旗標、G5/G6組合碼與兩市三池分母均保存。')
display(result[[KEY,'candidate_name','county_city','office_type','cluster_profile_id']].head(10))


210人一對一連結；已知值、缺值、資格旗標、G5/G6組合碼與兩市三池分母均保存。


,candidate_key,candidate_name,county_city,office_type,cluster_profile_id
0,2026-C1-001,郭璽,臺北市,直轄市長,P00
1,2026-C1-002,沈伯洋,臺北市,直轄市長,P01
2,2026-C1-003,蕭文乾,臺北市,直轄市長,P00
3,2026-C1-004,唐新民,臺北市,直轄市長,P00
4,2026-C1-005,蔣萬安,臺北市,直轄市長,P01
5,2026-C1-006,林志成,臺北市,直轄市長,P00
6,2026-C1-019,張靜,高雄市,直轄市長,P00
7,2026-C1-020,賴瑞隆,高雄市,直轄市長,P00
8,2026-C1-021,王肇民,高雄市,直轄市長,P10
9,2026-C1-022,柯志恩,高雄市,直轄市長,P01


In [4]:
OUT='data/processed/tk_g5_g6_characteristics_20261110.csv'
out=(ROOT/OUT).resolve()
assert out.is_relative_to((ROOT/'data/processed').resolve()) and out not in paths.values()
if out.exists():
    existing=pd.read_csv(out,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(result,existing)
else:
    with out.open('x',encoding='utf-8',newline='') as f: result.to_csv(f,index=False)
reread=pd.read_csv(out,dtype=str,keep_default_na=False)
pd.testing.assert_frame_equal(result,reread)
assert all(sha(paths[p])==h for p,h in PINS.items())
print({'path':OUT,'rows':len(reread),'columns':len(reread.columns),'sha256':sha(out),
       'one_to_one_merge':True,'all_source_cells_preserved':True,'serialized_readback':True})


{'path': 'data/processed/tk_g5_g6_characteristics_20261110.csv', 'rows': 210, 'columns': 53, 'sha256': '9df3c70602e5ac1293422dfed1a4a008e07d7280b069225d30a3cb250000a8a0', 'one_to_one_merge': True, 'all_source_cells_preserved': True, 'serialized_readback': True}
